In [38]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import pandas as pd
import ast

In [40]:
df_sales_2025 = pd.read_csv('../data/raw/vg_sales_2025.csv', index_col=False, parse_dates=["release_date", "last_update"])
pd.set_option('display.max_rows', None)

In [41]:
# Limpieza - datos nulos, duplicados, y preparación


# 0 - Creación de funciones y eliminación de columnas innecesarias y duplicados.

def null_values_fields(dataframe):
    null_counts = dataframe.isna().sum()
    return null_counts[null_counts > 0]

def sales_null_filler(total_sales, sales):
    to_fill = sales.pop(0)
    to_fill_clean = to_fill.fillna(
        total_sales
        - sum(x.fillna(0) for x in sales)
    )
    sales.append(to_fill_clean)
    return sales, to_fill_clean


# Dropeamos columna que no podemos utilizar para ningún tipo de análisis. 
# 'img' no nos sirve, no tenemos acceso a las imágenes. 'last_update' podría habernos dado información interesante para el análisis, pero el 77% de los datos útiles es faltante en esa columna.
df_sales_2025 = df_sales_2025.drop(columns=['img', 'last_update'])

df_sales_2025 = df_sales_2025.drop_duplicates(subset=['title', 'console'])


# 1 - Manejo de nulos de columnas de ventas

total_sales_null = df_sales_2025[df_sales_2025["total_sales"].isna()]
print(f"1. Nulos con venta en nulo: \n{null_values_fields(total_sales_null)}")
df_sales_2025 = df_sales_2025.dropna(subset=['total_shipped', 'total_sales'], how='all')

sales = [df_sales_2025['other_sales'], df_sales_2025['pal_sales'], df_sales_2025['na_sales'], df_sales_2025['jp_sales']]
df_sales_2025 = df_sales_2025[(df_sales_2025['total_sales'] > 0) | (df_sales_2025['total_shipped'] > 0)]

print()
print(f"2. Nulos de dataset después del paso 0: \n{null_values_fields(df_sales_2025)}")

# Utilizamos la función sales_null_filler para rellenar los valores nulos de cada columna de ventas sin asumir que valen 0.

# df_sales_2025 = df_sales_2025.sort_values(by=['total_shipped', 'total_sales'], ascending=False)

df_notna = df_sales_2025[df_sales_2025['total_sales'].notna()].copy()
total_sales_notna = df_notna['total_sales']

sales, df_notna['other_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['pal_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['na_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['jp_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)

df_sales_2025.loc[df_notna.index, ["other_sales", "pal_sales", "na_sales", "jp_sales"]] = \
    df_notna[["other_sales", "pal_sales", "na_sales", "jp_sales"]]

print()
print(f"3. Nulos de dataset después del paso 1: \n{null_values_fields(df_sales_2025)}")


# 2 - Manejo de nulos de las columnas developer, release_date y critic_score


# Al ser pocos valores nulos en la columna 'developer', se hizo una búsqueda para encontrar los valores faltantes, utilizando como fuente páginas oficiales de documentación de los juegos.
corrections = {
    'Gourmet Chef: Cook Your Way to Fame': 'Creative Patterns',
    'Wordmaster': 'Sarbakan'
}
df_sales_2025["developer"] = df_sales_2025['developer'].fillna(
    df_sales_2025['title'].map(corrections)
)


with open("../data/date_corrections.txt", "r", encoding="utf-8") as file:
    date_corrections = file.read()

date_corrections_dict = ast.literal_eval(date_corrections)

df_sales_2025['release_date'] = df_sales_2025['release_date'].fillna(
    df_sales_2025['title'].map(date_corrections_dict)
)

print()
print(f"4. Nulos de dataset después del paso 2: \n{null_values_fields(df_sales_2025)}")

# Después de investigar acerca de las filas que tenían release_date nulo, se descubrió que un 75% de las filas correspondían a juegos de PC.

pc = df_sales_2025["console"] == "PC"

print()
print(df_sales_2025.loc[pc, "release_date"].isna().mean())

# Debido a que un 17% de los valores de release_date de los juegos de PC están vacíos, se decidió no eliminar las filas vacías de fecha por ahora, y en cambio hacerlo solo si necesitamos un análisis de tiempo.

# Finalmente, creamos una columna que contenga el año de salida de cada juego en lugar de la fecha completa.

df_sales_2025['year'] = df_sales_2025['release_date'].dt.year
df_sales_2025['year'] = df_sales_2025['year'].astype('Int64')


1. Nulos con venta en nulo: 
developer           13
vg_score         46374
critic_score     45432
user_score       47822
total_shipped    43425
total_sales      48078
na_sales         48078
jp_sales         48078
pal_sales        48078
other_sales      48078
release_date      9251
dtype: int64

2. Nulos de dataset después del paso 0: 
developer            2
vg_score         21005
critic_score     17225
user_score       21837
total_shipped    17485
total_sales       4637
na_sales          9699
jp_sales         15703
pal_sales        10379
other_sales       8021
release_date       531
dtype: int64

3. Nulos de dataset después del paso 1: 
developer            2
vg_score         21005
critic_score     17225
user_score       21837
total_shipped    17485
total_sales       4637
na_sales          4637
jp_sales          4637
pal_sales         4637
other_sales       4637
release_date       531
dtype: int64

4. Nulos de dataset después del paso 2: 
vg_score         21005
critic_score     17225
u

In [ ]:
# Desde aquí, se hará una separación del dataset en 2. Una parte tendrá todos los registros que tengan 'Series' en su campo console, ya que representan sagas completas de juegos.
# El otro dataset será el de los juegos individuales o 'All' (que representan al juego en todas sus plataformas en lugar de una específica)

df_sales_series = df_sales_2025[df_sales_2025['console'] == 'Series'].copy()
df_sales_ind = df_sales_2025[~df_sales_2025.index.isin(df_sales_series.index)].copy()

print(f"4. Nulos de dataset individual: \n{null_values_fields(df_sales_ind)}")
print()
print(f"4. Nulos de dataset series: \n{null_values_fields(df_sales_series)}")
print()

# Debido a que el dataset Series tiene varias columnas completamente vacías, se eliminarán dichas columnas. Además, se eliminan las columnas innecesarias 'console' y 'developer'.

df_sales_series = df_sales_series.dropna(axis=1, how='all')
df_sales_series = df_sales_series.drop(columns=['console', 'developer'])

print(df_sales_ind["user_score"].isna().mean())
print(df_sales_ind["critic_score"].isna().mean())
print(df_sales_ind["vg_score"].isna().mean())

df_sales_ind = df_sales_ind.drop(columns=['vg_score', 'user_score', 'developer'])

# Debido a la alta proporción de datos faltantes de las tres columnas de puntuación, se decidió eliminar las columnas del puntaje de vgchartz y user, pero dejar las de critic por si se quieren analizar.
# Nota después del EDA: Se elimina la columna de 'developer' ya que no se usó para ningún análisis

df_sales_ind["units_reported"] = (
    df_sales_ind["total_sales"]
    .fillna(df_sales_ind["total_shipped"])
)

df_sales_ind = df_sales_ind.sort_values('units_reported', ascending=False)
df_sales_series = df_sales_series.sort_values('total_shipped', ascending=False)


# 3 - Manejo de valores inconsistentes

# Al investigar con 'print(df_sales_ind['console'].unique())', se descubre que el la inusual plataforma 'WW'.
# Al investigarlo, se descubre que en realidad 'WW' debería ser 'Wii', por lo que se corrige.
print(df_sales_ind[df_sales_2025['console'] == 'WW'])
df_sales_ind.loc[df_sales_ind['title'] == 'Karaoke Joysound Wii', 'console'] = 'Wii'

# Después de revisar el resto de columnas, no se encontraron más valores inconsistentes.

4. Nulos de dataset individual: 
vg_score         20522
critic_score     16742
user_score       21354
total_shipped    17485
total_sales       4154
na_sales          4154
jp_sales          4154
pal_sales         4154
other_sales       4154
release_date       492
year               492
dtype: int64

4. Nulos de dataset series: 
vg_score        483
critic_score    483
user_score      483
total_sales     483
na_sales        483
jp_sales        483
pal_sales       483
other_sales     483
dtype: int64

0.9868293359212533
0.7736956421276399
0.9483802393825962
                      title console genre    publisher  critic_score  \
21044  Karaoke Joysound Wii      WW  Misc  Hudson Soft           NaN   

       total_shipped  total_sales  na_sales  jp_sales  pal_sales  other_sales  \
21044            NaN         0.25       0.0      0.25        0.0          0.0   

      release_date  year  units_reported  
21044   2009-07-29  2009            0.25  


C:\Users\dschu\AppData\Local\Temp\ipykernel_18308\1493932060.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(df_sales_ind[df_sales_2025['console'] == 'WW'])


In [43]:
print(df_sales_ind.info())
print(df_sales_series.info())

# El dataset finaliza la limpeza con 21.640 filas restantes de 67.172 originales. Aunque solo quedan el 32,2% de las filas originales, 
# Más de 21000 filas siguen siendo datos suficientes para el análisis que se desea realizar.

df_sales_ind.to_csv('../data/processed/sales_ind_clean.csv', index=False)
df_sales_series.to_csv('../data/processed/sales_series_clean.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 21639 entries, 15594 to 66799
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           21639 non-null  object        
 1   console         21639 non-null  object        
 2   genre           21639 non-null  object        
 3   publisher       21639 non-null  object        
 4   critic_score    4897 non-null   float64       
 5   total_shipped   4154 non-null   float64       
 6   total_sales     17485 non-null  float64       
 7   na_sales        17485 non-null  float64       
 8   jp_sales        17485 non-null  float64       
 9   pal_sales       17485 non-null  float64       
 10  other_sales     17485 non-null  float64       
 11  release_date    21147 non-null  datetime64[ns]
 12  year            21147 non-null  Int64         
 13  units_reported  21639 non-null  float64       
dtypes: Int64(1), datetime64[ns](1), float64(8), object(4)
m